# 🔐 Day 02a: File Permissions — chmod, chown, umask & setuid

---

## 🎯 What You'll Master Today
- Linux permission model (rwx for user/group/others)
- chmod (symbolic & octal), chown, chgrp
- umask — default permission mask
- Special permissions: setuid, setgid, sticky bit

---

```
╔══════════════════════════════════════════════════════════════╗
║           LINUX PERMISSION MODEL                             ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  -rwxr-xr--  1  user  group  4096  Jan 1  file.txt          ║
║  │└┬┘└┬┘└┬┘                                                 ║
║  │ │  │  └── Others: r-- (read only)                        ║
║  │ │  └───── Group:  r-x (read + execute)                   ║
║  │ └──────── Owner:  rwx (read + write + execute)           ║
║  └────────── Type:   - (file), d (dir), l (link)            ║
║                                                              ║
║  Octal:  r=4  w=2  x=1  │  rwx=7  rw-=6  r-x=5  r--=4    ║
║  Above = 754             │  chmod 754 file.txt               ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Permission Calculator
# ============================================

def perm_to_octal(perm_str: str) -> str:
    """Convert rwxr-xr-- to 754."""
    if len(perm_str) not in (9, 10):
        return "Invalid"
    
    # Strip type character if present
    p = perm_str[-9:]
    
    octal = ""
    for i in range(0, 9, 3):
        val = 0
        if p[i] == 'r':   val += 4
        if p[i+1] == 'w': val += 2
        if p[i+2] == 'x': val += 1
        octal += str(val)
    
    return octal


def octal_to_perm(octal: str) -> str:
    """Convert 754 to rwxr-xr--."""
    result = ""
    for digit in octal:
        val = int(digit)
        result += 'r' if val & 4 else '-'
        result += 'w' if val & 2 else '-'
        result += 'x' if val & 1 else '-'
    return result


# Common permission patterns
patterns = [
    ("755", "Executables, directories"),
    ("644", "Regular files"),
    ("600", "Private files (SSH keys)"),
    ("777", "Full access (AVOID!)"),
    ("750", "Group-shared executables"),
    ("700", "Private directories"),
]

print("  Permission Patterns")
print(f"  {'Octal':<8} {'Symbolic':<12} {'Use Case'}")
print("  " + "─" * 50)
for octal, use in patterns:
    symbolic = octal_to_perm(octal)
    print(f"  {octal:<8} {symbolic:<12} {use}")

# Quick converter
print("\n  Quick Convert:")
print(f"    rwxr-xr-- → {perm_to_octal('rwxr-xr--')}")
print(f"    644       → {octal_to_perm('644')}")

In [ ]:
# ============================================
# 2. umask Calculator
# ============================================

def umask_result(umask_val: str):
    """
    Calculate actual permissions from umask.
    Default: files=666, directories=777
    Actual = default - umask (bitwise)
    """
    umask_int = int(umask_val, 8)
    
    file_default = 0o666
    dir_default = 0o777
    
    file_actual = file_default & ~umask_int
    dir_actual = dir_default & ~umask_int
    
    print(f"  umask = {umask_val}")
    print(f"  " + "─" * 40)
    print(f"  Files:       666 & ~{umask_val} = {oct(file_actual)[2:]:>03} ({octal_to_perm(f'{file_actual:03o}')})")
    print(f"  Directories: 777 & ~{umask_val} = {oct(dir_actual)[2:]:>03} ({octal_to_perm(f'{dir_actual:03o}')})")


# Common umask values
print("  umask Effects:")
print("  " + "═" * 45)
for mask in ["022", "027", "077", "002"]:
    umask_result(mask)
    print()

print("  💡 Default umask: 022 → files=644, dirs=755")
print("  💡 Secure umask: 077 → files=600, dirs=700")

In [ ]:
# ============================================
# 3. Special Permissions Explained
# ============================================

special_perms = [
    {
        "name": "setuid (SUID)",
        "octal": "4xxx",
        "symbol": "s in owner execute (rws)",
        "effect": "Process runs as FILE OWNER, not caller",
        "example": "/usr/bin/passwd — any user can change their password\n"
                   "                     (runs as root to edit /etc/shadow)",
        "danger": "SUID root binary = potential privilege escalation!",
        "set_cmd": "chmod u+s file  OR  chmod 4755 file",
    },
    {
        "name": "setgid (SGID)",
        "octal": "2xxx",
        "symbol": "s in group execute (r-s)",
        "effect": "On file: runs as file's group. On dir: new files inherit dir's group",
        "example": "Shared project dir — all files auto-owned by 'devteam' group",
        "danger": "Less risky than SUID, but audit on executables",
        "set_cmd": "chmod g+s dir  OR  chmod 2755 dir",
    },
    {
        "name": "Sticky Bit",
        "octal": "1xxx",
        "symbol": "t in others execute (rwt)",
        "effect": "Only file owner can delete files in directory",
        "example": "/tmp — everyone can write, only owner can delete own files",
        "danger": "Essential for shared dirs, prevents malicious deletion",
        "set_cmd": "chmod +t dir  OR  chmod 1777 dir",
    },
]

for perm in special_perms:
    print(f"  {'═' * 55}")
    print(f"  {perm['name']} ({perm['octal']})")
    print(f"  {'═' * 55}")
    print(f"    Symbol:  {perm['symbol']}")
    print(f"    Effect:  {perm['effect']}")
    print(f"    Example: {perm['example']}")
    print(f"    ⚠️ Note:  {perm['danger']}")
    print(f"    Set:     {perm['set_cmd']}")
    print()

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What does chmod 755 mean? | Owner:rwx, Group:r-x, Others:r-x. Standard for executables/dirs |
| 2 | What is umask 022? | Files created as 644, dirs as 755. Blocks write for group/others |
| 3 | What is setuid and why is it dangerous? | Process runs as file owner (often root). Vulnerable binary = root access |
| 4 | Why does /tmp have sticky bit? | Everyone can write files, but only owner can delete their own files |
| 5 | chown vs chmod? | chown: change ownership. chmod: change permissions (rwx) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • rwx triplet: owner | group | others (9 bits)         │
## │  • Octal: r=4, w=2, x=1. 755 = rwxr-xr-x               │
## │  • umask: subtracts from default (666/777)               │
## │  • SUID: runs as file owner — audit carefully            │
## │  • Sticky bit: protects shared dirs (/tmp)               │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Shell Scripting** — variables, conditionals, loops, grep/sed/awk